# end-grad-default-ones-like — faded example 1: resolve_end_grad: None → ones_like, Else Unbox

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `end-grad-default-ones-like`. Running the beacon reports progress on the `Backprop: end-grad ones_like default` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: end-grad ones_like default` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`end-grad-default-ones-like`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "end-grad-default-ones-like"
DD_SUBTOPIC = "Backprop: end-grad ones_like default"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The backward entry-point convention is: when no seed is supplied, produce `ones_like(end_node.array)` (equivalent to summing the end node and differentiating that scalar). When a seed is explicitly supplied, unbox it from the MiniTensor wrapper and verify the shape matches before passing it downstream.

## Faded exercise 1

Implement `resolve_end_grad(end_node, end_grad)`. If `end_grad is None`, return `t.ones_like(end_node.array)`. Otherwise assert `end_grad.array.shape == end_node.array.shape` (raise `AssertionError` with a message listing both shapes) and return `end_grad.array`.

**Fill in:** When end_grad is not None, assert the shapes match (raising AssertionError with both shapes in the message) and return end_grad.array; when end_grad is None, return t.ones_like(end_node.array).

In [ ]:
import torch as t

class MiniTensor:
    def __init__(self, array):
        self.array = array

def resolve_end_grad(end_node: MiniTensor, end_grad):
    raise NotImplementedError()  # TODO: When end_grad is not None, assert the shapes match (raising AssertionError with both shapes in the message) and return end_grad.array; when end_grad is None, return t.ones_like(end_node.array).


def _test():
    import torch as t
    class MiniTensor:
        def __init__(self, array):
            self.array = array
    node = MiniTensor(t.zeros(5))
    # path 1: None -> ones_like
    seed = resolve_end_grad(node, None)
    assert seed.shape == (5,)
    assert (seed == 1.0).all()
    assert seed.dtype == t.float32
    # path 2: explicit, shape match
    eg = MiniTensor(t.arange(5, dtype=t.float32))
    result = resolve_end_grad(node, eg)
    assert t.allclose(result, eg.array)
    # path 3: shape mismatch raises AssertionError
    bad = MiniTensor(t.zeros(3))
    try:
        resolve_end_grad(node, bad)
        assert False
    except AssertionError:
        pass


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class MiniTensor:
    def __init__(self, array):
        self.array = array

def resolve_end_grad(end_node: MiniTensor, end_grad):
    if end_grad is None:
        return t.ones_like(end_node.array)
    assert end_grad.array.shape == end_node.array.shape, (
        f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
        f'end_node shape {tuple(end_node.array.shape)}'
    )
    return end_grad.array
```
</details>